# 🚀 NewsBlurb Quant Model Trainer

This notebook fine-tunes **Llama-3-8b** on your custom **NewsBlurb Quant Dataset**.
It uses [Unsloth](https://github.com/unslothai/unsloth) for 2x faster training and 60% less memory usage.

In [ ]:
%%capture
# 1. Install Unsloth & Dependencies
import torch
major_version, minor_version = torch.cuda.get_device_capability()
# Must install separately due to Colab dependency conflicts
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    !pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# 2. Load Base Model (Llama-3-8b-Instruct)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# 3. Upload your Training Data
from google.colab import files
print("Please upload your 'training_data.jsonl' file now...")
uploaded = files.upload()

In [ ]:
# 4. Prepare Dataset
from datasets import Dataset
import json

# Standard Alpaca Format
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = [""] * len(instructions) # We put everything in instruction usually, or split it
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # For our use case, 'instruction' contains the JSON data
        text = alpaca_prompt.format("Analyze this market data as a Quant Analyst.", instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Load the JSONL
import pandas as pd
df = pd.read_json("training_data.jsonl", lines=True)
dataset = Dataset.from_pandas(df)
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 5. Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Train for 60 steps (adjust based on dataset size)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 6. Run Inference (Test)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# IMPORTANT: Paste one of your JSON inputs here for testing
test_input = """
{
  "ticker": "AAPL",
  "price": 150.25,
  "macro": {
     "vix": 28.5,
     "oil": 85.0
  },
  "correlation_sp500": 0.92
}
"""

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Analyze this market data as a Quant Analyst.", # instruction
        test_input, # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
result = tokenizer.batch_decode(outputs)
print(result[0])

In [ ]:
# 7. Save Model (GGUF)
# usage: model.save_pretrained_gguf("model_name", tokenizer, quantization_method = "f16")
model.save_pretrained("newsblurb_quant_model")
print("Model saved to 'newsblurb_quant_model'")
# You can also push to Hugging Face Hub here